**Human in the Loop MiddleWare**

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode , tools_condition

from langgraph.types import Command,interrupt

from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)



class State(TypedDict):
    message:Annotated[list,add_messages]

graph_builder = StateGraph(State)

@tool
def human_assistance(query : str)->str:
    """request assistance from a human."""
    human_response = interrupt({"query":query})

tool = TavilySearch(max_results=2)
tools =[human_assistance,tool]

llm_with_tools=llm.bind_tools(tools)

def chatbot(state:State):
    # because we will be interuupting during tool execution,
    # we dsiable paralled tool calling to avoid repeating any
    # tool invocation when we resume.
    message=llm_with_tools.invoke(state['messages'])



